*Procesamiento de archivo TXT*

El objetivo de este proyecto es efectuar un Análisis Explotario de Datos y algunos pasos previos de limpieza tales como:  
1.- Eliminar rellenar nulos con la media de la columna, 
2.- Contar total de registros faltantes por fecha

In [2]:
import pandas as pd
df=pd.read_csv('25001.txt',
skiprows=21, #saltamos las primeras 18 lineas del archivo
skipfooter=2,#saltamos las ultimas 2 lineas
names=['FECHA','PRECIP','EVAP','TMAX','TMIN'] , #Nombres de las columnas 
sep=r'\s+',#Separamos las columnas correctamente 
na_values='Nulo', #Asignamos como nulos los datos que contengan 'Nulo'
encoding='latin1', # Soluciona el UnicodeDecodeError
    engine='python'
) 
df.head()

,FECHA,PRECIP,EVAP,TMAX,TMIN
0,03/01/1961,0.0,NaN,31.0,11.0
1,04/01/1961,0.0,NaN,32.0,13.0
2,05/01/1961,0.0,NaN,29.0,14.0
3,06/01/1961,0.0,NaN,24.0,14.5
4,07/01/1961,0.0,NaN,28.5,10.0


In [3]:
def registros_faltantes_por_fecha(df):
    
    # Conversion de columna FECHA a datetime y tomamos solamente la fecha 
    df['FECHA'] = pd.to_datetime(df['FECHA'], dayfirst=True)
    df['FECHA'] = df['FECHA'].dt.date 
    columnas_clima = ['PRECIP', 'EVAP', 'TMAX', 'TMIN']    # Sumamos la cantidad de valores nulos (NaN) por cada fila en las columnas meteorológicas
    df['VARIABLES_NULAS'] = df[columnas_clima].isnull().sum(axis=1)
    
    # Filtramos el dataframe para dejar unicamente las fechas que presenten al menos 1 registro faltante
    dias_incompletos = df[df['VARIABLES_NULAS'] > 0]
    tabla_resumen = dias_incompletos[['FECHA', 'VARIABLES_NULAS']]    # Creamos la tabla resumen solo con la fecha y la cantidad exacta de variables ausentes

    # Imprimimos el total de días afectados siguiendo tu formato
    print(f"Dias con registros nulos en la tabla: {len(tabla_resumen)}")
    return tabla_resumen

# Ejecucion del metodo
registros_faltantes_por_fecha(df)

Dias con registros nulos en la tabla: 310


,FECHA,VARIABLES_NULAS
0,1961-01-03,1
1,1961-01-04,1
2,1961-01-05,1
3,1961-01-06,1
4,1961-01-07,1
...,...,...
18926,2016-07-23,1
18928,2016-07-25,1
19208,2017-06-01,1
19325,2017-09-26,1


In [4]:
def contar_nulos_por_anio(df_temp):#NO SE SI ESTA BIEEEEEN
    """
    Función para contar cuántos datos faltan en cada columna, 
    separados por cada año del registro.
    """
    # Convertimos la columna de fecha a formato datetime
    df_temp['FECHA'] = pd.to_datetime(df_temp['FECHA'], dayfirst=True)
    # Creamos la columna 'Año' extrayendo el año de la columna 'FECHA'
    df_temp['Año'] = df_temp['FECHA'].dt.year
    # Definimos las columnas numéricas que queremos analizar
    cols_clima = ['PRECIP', 'EVAP', 'TMAX', 'TMIN']
    nulos_por_anio = df_temp[cols_clima].isnull().groupby(df_temp['Año']).sum()    # Calculamos los nulos por año (Usando df_temp)
    print("--- Cantidad de datos nulos detectados por Año ---")
    display(nulos_por_anio)
    print("\nTotal de nulos en todo el registro:")    # Opcional: Ver el total general de nulos antes de proceder (Usando df_temp)
    print(df_temp[cols_clima].isnull().sum())

    return df_temp
df = contar_nulos_por_anio(df)

--- Cantidad de datos nulos detectados por Año ---


,PRECIP,EVAP,TMAX,TMIN
Año,,,,
1961,0,59,0,0
1962,0,2,0,0
1963,0,4,0,0
1964,1,0,0,0
1965,0,4,0,0
1966,1,3,0,0
1967,0,2,0,0
1968,0,1,0,0
1969,30,1,1,1



Total de nulos en todo el registro:
PRECIP     40
EVAP      270
TMAX        5
TMIN        6
dtype: int64


In [5]:
def transformar_datos(df_temp):
    """
    Función modular para convertir tipos de datos y eliminar basura de texto.
    """
    # 1. Convertimos la fecha usando errors='coerce' como prometiste en tu texto
    # Esto convertirá cualquier texto que no sea fecha (como títulos colados) en NaT (Not a Time)
    df_temp['FECHA'] = pd.to_datetime(df_temp['FECHA'], dayfirst=True, errors='coerce')

    # 2. Aplicamos dropna() para eliminar esas filas que resultaron ser texto basura (NaT)
    # Solo borramos filas si la FECHA está vacía/rota
    df_temp = df_temp.dropna(subset=['FECHA'])

    # 3. Convertimos las columnas de medición a numéricas
    cols_num = ['PRECIP', 'EVAP', 'TMAX', 'TMIN']
    
    # También es excelente práctica agregar errors='coerce' aquí por si se coló algún espacio o letra
    df_temp[cols_num] = df_temp[cols_num].apply(pd.to_numeric, errors='coerce') 
    
    return df_temp

# Llamada a la función
df = transformar_datos(df)
df.head()

,FECHA,PRECIP,EVAP,TMAX,TMIN,VARIABLES_NULAS,Año
0,1961-01-03,0.0,NaN,31.0,11.0,1,1961
1,1961-01-04,0.0,NaN,32.0,13.0,1,1961
2,1961-01-05,0.0,NaN,29.0,14.0,1,1961
3,1961-01-06,0.0,NaN,24.0,14.5,1,1961
4,1961-01-07,0.0,NaN,28.5,10.0,1,1961


In [6]:
def aplicar_limpieza_nulos(df_temp):
    """
    Rellena los espacios vacíos con el promedio de la columna.
    """
    cols_num = ['PRECIP', 'EVAP', 'TMAX', 'TMIN']
    
    # Rellenamos nulos con la media de cada columna 
    df_temp[cols_num] = df_temp[cols_num].fillna(df_temp[cols_num].mean())
    
    return df_temp

# Aplicamos la limpieza
df = aplicar_limpieza_nulos(df)
# Verificamos que ya no existan nulos
df.isnull().sum()

FECHA              0
PRECIP             0
EVAP               0
TMAX               0
TMIN               0
VARIABLES_NULAS    0
Año                0
dtype: int64

FALTAAAAA El código está documentado, es ordenado y presenta las tablas resultantes de manera clara y comprensible para un usuario no técnico.